In [1]:
%pip install -q \
    "qiskit>=2.0" \
    "qiskit-algorithms>=0.4" \
    "qbraid>=0.12" \
    pandas \
    openpyxl

Note: you may need to restart the kernel to use updated packages.


In [1]:
import math
import time
from itertools import combinations, permutations
from collections import defaultdict

import numpy as np
import pandas as pd

from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorSampler

from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA


# ============================================================
# REPRODUCIBILITY
# ============================================================

RANDOM_SEEDS = [7, 19, 42]

np.random.seed(42)


# ============================================================
# QAOA SETTINGS
# ============================================================

# Start with p = 1.
# Later benchmark p = 1, 2, 3.
QAOA_REPS = 1

# Number of measurements of final quantum state
SHOTS = 4096

# Classical parameter optimization iterations
MAXITER = 200

# CVaR focuses QAOA on the low-energy tail.
CVAR_ALPHA = 0.25

# Maximum factories in one quantum route problem.
#
# m factories -> m^2 qubits.
#
# 2 factories -> 4 qubits
# 3 factories -> 9 qubits
# 4 factories -> 16 qubits
MAX_QUANTUM_FACTORIES = 4


print("Quantum environment ready.")
print("QAOA reps:", QAOA_REPS)
print("Shots:", SHOTS)
print("Maximum factories per quantum problem:", MAX_QUANTUM_FACTORIES)

Quantum environment ready.
QAOA reps: 1
Shots: 4096
Maximum factories per quantum problem: 4


In [2]:
# ============================================================
# LOAD THE SAME DATASET USED BY CLASSICAL MODEL
# ============================================================

DATA_FILE = "QSOLVE_Quantum_Ready_Dataset.xlsx"


# ------------------------------------------------------------
# Nodes and demand
# ------------------------------------------------------------

nodes_df = pd.read_excel(
    DATA_FILE,
    sheet_name="Nodes_Demand"
)


# ------------------------------------------------------------
# Distance matrix
# ------------------------------------------------------------

DIST = pd.read_excel(
    DATA_FILE,
    sheet_name="Distance_Matrix",
    index_col=0
)

DIST = DIST.astype(float)


# ------------------------------------------------------------
# Travel-time matrix
# ------------------------------------------------------------

TIME = pd.read_excel(
    DATA_FILE,
    sheet_name="Travel_Time_Matrix",
    index_col=0
)

TIME = TIME.astype(float)


# ------------------------------------------------------------
# Route allowed matrix
# ------------------------------------------------------------

ROUTE_ALLOWED = pd.read_excel(
    DATA_FILE,
    sheet_name="Route_Allowed",
    index_col=0
)

ROUTE_ALLOWED = ROUTE_ALLOWED.astype(int)


# ------------------------------------------------------------
# Parameters
# ------------------------------------------------------------

parameters_df = pd.read_excel(
    DATA_FILE,
    sheet_name="Parameters"
)

parameter_map = dict(
    zip(
        parameters_df["Parameter"],
        parameters_df["Value"]
    )
)


# ============================================================
# EXTRACT MODEL PARAMETERS
# ============================================================

DEPOT = "Karatina"

DEPOT_STOCK = float(
    parameter_map["Depot stock"]
)

TRUCK_CAPACITY = float(
    parameter_map["Truck capacity"]
)

TRANSPORT_RATE = float(
    parameter_map["Transport rate"]
)

MAX_ROUTE_TIME = float(
    parameter_map["Maximum route duration"]
)

DISPATCH_DAYS = int(
    parameter_map["Dispatch period"]
)


# ============================================================
# EXTRACT FACTORY DEMAND
# ============================================================

factory_rows = nodes_df[
    nodes_df["Type"].str.lower() == "factory"
]

FACTORIES = factory_rows[
    "Location"
].tolist()

DEMAND = dict(
    zip(
        factory_rows["Location"],
        factory_rows["Demand_tonnes"].astype(float)
    )
)


TOTAL_DEMAND = sum(
    DEMAND.values()
)


# ============================================================
# VALIDATE DATA
# ============================================================

print("=" * 70)
print("QSOLVE QUANTUM DATA")
print("=" * 70)

print("\nFactories:")
for factory in FACTORIES:
    print(
        f"{factory:<12}: "
        f"{DEMAND[factory]:>8.3f} tonnes"
    )

print("\nTotal demand      :", TOTAL_DEMAND)
print("Depot stock       :", DEPOT_STOCK)
print("Truck capacity    :", TRUCK_CAPACITY)
print("Transport rate    :", TRANSPORT_RATE)
print("Maximum route time:", MAX_ROUTE_TIME)

assert abs(TOTAL_DEMAND - 233.700) < 1e-6
assert DEPOT_STOCK == 410.0
assert TRUCK_CAPACITY == 10.0

print("\n✓ Dataset successfully loaded.")

QSOLVE QUANTUM DATA

Factories:
Ragati      :   28.380 tonnes
Nguguini    :   73.092 tonnes
Kiangai     :   36.489 tonnes
Thunguri    :   37.170 tonnes
Kianjega    :   24.522 tonnes
Chema       :   10.848 tonnes
Mukangu     :   23.199 tonnes

Total demand      : 233.7
Depot stock       : 410.0
Truck capacity    : 10.0
Transport rate    : 16.0
Maximum route time: 8.0

✓ Dataset successfully loaded.


In [3]:
# ============================================================
# LOAD THE SAME DATASET USED BY CLASSICAL MODEL
# ============================================================

DATA_FILE = "QSOLVE_Quantum_Ready_Dataset.xlsx"


# ------------------------------------------------------------
# Nodes and demand
# ------------------------------------------------------------

nodes_df = pd.read_excel(
    DATA_FILE,
    sheet_name="Nodes_Demand"
)


# ------------------------------------------------------------
# Distance matrix
# ------------------------------------------------------------

DIST = pd.read_excel(
    DATA_FILE,
    sheet_name="Distance_Matrix",
    index_col=0
)

DIST = DIST.astype(float)


# ------------------------------------------------------------
# Travel-time matrix
# ------------------------------------------------------------

TIME = pd.read_excel(
    DATA_FILE,
    sheet_name="Travel_Time_Matrix",
    index_col=0
)

TIME = TIME.astype(float)


# ------------------------------------------------------------
# Route allowed matrix
# ------------------------------------------------------------

ROUTE_ALLOWED = pd.read_excel(
    DATA_FILE,
    sheet_name="Route_Allowed",
    index_col=0
)

ROUTE_ALLOWED = ROUTE_ALLOWED.astype(int)


# ------------------------------------------------------------
# Parameters
# ------------------------------------------------------------

parameters_df = pd.read_excel(
    DATA_FILE,
    sheet_name="Parameters"
)

parameter_map = dict(
    zip(
        parameters_df["Parameter"],
        parameters_df["Value"]
    )
)


# ============================================================
# EXTRACT MODEL PARAMETERS
# ============================================================

DEPOT = "Karatina"

DEPOT_STOCK = float(
    parameter_map["Depot stock"]
)

TRUCK_CAPACITY = float(
    parameter_map["Truck capacity"]
)

TRANSPORT_RATE = float(
    parameter_map["Transport rate"]
)

MAX_ROUTE_TIME = float(
    parameter_map["Maximum route duration"]
)

DISPATCH_DAYS = int(
    parameter_map["Dispatch period"]
)


# ============================================================
# EXTRACT FACTORY DEMAND
# ============================================================

factory_rows = nodes_df[
    nodes_df["Type"].str.lower() == "factory"
]

FACTORIES = factory_rows[
    "Location"
].tolist()

DEMAND = dict(
    zip(
        factory_rows["Location"],
        factory_rows["Demand_tonnes"].astype(float)
    )
)


TOTAL_DEMAND = sum(
    DEMAND.values()
)


# ============================================================
# VALIDATE DATA
# ============================================================

print("=" * 70)
print("QSOLVE QUANTUM DATA")
print("=" * 70)

print("\nFactories:")
for factory in FACTORIES:
    print(
        f"{factory:<12}: "
        f"{DEMAND[factory]:>8.3f} tonnes"
    )

print("\nTotal demand      :", TOTAL_DEMAND)
print("Depot stock       :", DEPOT_STOCK)
print("Truck capacity    :", TRUCK_CAPACITY)
print("Transport rate    :", TRANSPORT_RATE)
print("Maximum route time:", MAX_ROUTE_TIME)

assert abs(TOTAL_DEMAND - 233.700) < 1e-6
assert DEPOT_STOCK == 410.0
assert TRUCK_CAPACITY == 10.0

print("\n✓ Dataset successfully loaded.")

QSOLVE QUANTUM DATA

Factories:
Ragati      :   28.380 tonnes
Nguguini    :   73.092 tonnes
Kiangai     :   36.489 tonnes
Thunguri    :   37.170 tonnes
Kianjega    :   24.522 tonnes
Chema       :   10.848 tonnes
Mukangu     :   23.199 tonnes

Total demand      : 233.7
Depot stock       : 410.0
Truck capacity    : 10.0
Transport rate    : 16.0
Maximum route time: 8.0

✓ Dataset successfully loaded.


In [4]:
print("DISTANCE MATRIX (km)")
display(DIST)

print("\nTRAVEL-TIME MATRIX (hours)")
display(TIME)

print("\nROUTE ALLOWED MATRIX")
display(ROUTE_ALLOWED)

DISTANCE MATRIX (km)


,Karatina,Ragati,Nguguini,Kiangai,Thunguri,Kianjega,Chema,Mukangu
From/To,,,,,,,,
Karatina,0.00,11.82,18.92,20.02,23.13,13.11,11.95,10.08
Ragati,11.82,0.00,15.86,29.64,34.81,25.51,11.53,21.86
Nguguini,18.92,15.86,0.00,19.08,40.65,20.23,9.84,24.95
Kiangai,20.02,29.64,19.08,0.00,38.90,7.71,21.17,21.38
Thunguri,23.13,34.81,40.65,38.90,0.00,32.26,33.69,24.62
Kianjega,13.11,25.51,20.23,7.71,32.26,0.00,19.38,14.75
Chema,11.95,11.53,9.84,21.17,33.69,19.38,0.00,18.26
Mukangu,10.08,21.86,24.95,21.38,24.62,14.75,18.26,0.00



TRAVEL-TIME MATRIX (hours)


,Karatina,Ragati,Nguguini,Kiangai,Thunguri,Kianjega,Chema,Mukangu
From/To,,,,,,,,
Karatina,0.00,0.27,0.44,0.29,0.52,0.14,0.28,0.22
Ragati,0.27,0.00,0.38,0.55,0.79,0.42,0.28,0.47
Nguguini,0.44,0.38,0.00,0.44,0.92,0.43,0.23,0.57
Kiangai,0.29,0.55,0.44,0.00,0.79,0.15,0.51,0.43
Thunguri,0.52,0.79,0.92,0.79,0.00,0.63,0.75,0.56
Kianjega,0.14,0.42,0.43,0.15,0.63,0.00,0.35,0.28
Chema,0.28,0.28,0.23,0.51,0.75,0.35,0.00,0.41
Mukangu,0.22,0.47,0.57,0.43,0.56,0.28,0.41,0.00



ROUTE ALLOWED MATRIX


,Karatina,Ragati,Nguguini,Kiangai,Thunguri,Kianjega,Chema,Mukangu
From/To,,,,,,,,
Karatina,0,1,1,1,1,1,1,1
Ragati,1,0,1,1,1,1,1,1
Nguguini,1,1,0,1,1,1,1,1
Kiangai,1,1,1,0,1,1,1,1
Thunguri,1,1,1,1,0,1,1,1
Kianjega,1,1,1,1,1,0,1,1
Chema,1,1,1,1,1,1,0,1
Mukangu,1,1,1,1,1,1,1,0


In [5]:
# ============================================================
# CLASSICAL CAPACITY DECOMPOSITION
# ============================================================

EPS = 1e-9


def split_full_and_residual_demand(
    demand,
    capacity
):
    """
    Separate each factory demand into:
      1. full 10-tonne trips
      2. residual demand

    Returns:
        single_factory_trips
        residual_demand
    """

    single_factory_trips = []

    residual_demand = {}

    for factory, qty in demand.items():

        full_trips = int(
            math.floor(
                (qty + EPS) / capacity
            )
        )

        # Complete 10-t trips
        for _ in range(full_trips):

            single_factory_trips.append({
                factory: capacity
            })

        remainder = (
            qty
            - full_trips * capacity
        )

        residual_demand[factory] = round(
            max(0.0, remainder),
            3
        )

    return (
        single_factory_trips,
        residual_demand
    )


single_trips, residual = (
    split_full_and_residual_demand(
        DEMAND,
        TRUCK_CAPACITY
    )
)


print("=" * 70)
print("FULL 10-TONNE TRIPS")
print("=" * 70)

print(
    "Number of full trips:",
    len(single_trips)
)


print("\nResidual quantities:")

for factory, qty in residual.items():

    print(
        f"{factory:<12}: "
        f"{qty:>7.3f} t"
    )


print(
    "\nTotal residual =",
    sum(residual.values()),
    "tonnes"
)

FULL 10-TONNE TRIPS
Number of full trips: 20

Residual quantities:
Ragati      :   8.380 t
Nguguini    :   3.092 t
Kiangai     :   6.489 t
Thunguri    :   7.170 t
Kianjega    :   4.522 t
Chema       :   0.848 t
Mukangu     :   3.199 t

Total residual = 33.7 tonnes


In [6]:
# ============================================================
# CREATE MULTI-FACTORY TRIP GROUPS
# ============================================================

def create_multifactory_groups(
    residual_demand,
    distance_matrix,
    capacity=10.0,
    max_factories=4
):
    """
    Hybrid classical clustering.

    1. Start with factory having largest residual demand.
    2. Fill remaining truck capacity.
    3. Select the nearest factory with residual demand.
    4. Continue until:
         - capacity is full, or
         - max_factories reached.

    Quantum QAOA later optimizes the ORDER.
    """

    remaining = residual_demand.copy()

    groups = []


    while (
        sum(remaining.values()) > EPS
    ):

        active = [
            f
            for f, qty in remaining.items()
            if qty > EPS
        ]

        if not active:
            break


        # Start at factory with largest demand
        current = max(
            active,
            key=lambda f: remaining[f]
        )


        remaining_capacity = capacity

        trip = {}


        while (
            remaining_capacity > EPS
            and len(trip) < max_factories
        ):

            if remaining[current] > EPS:

                delivery = min(
                    remaining_capacity,
                    remaining[current]
                )

                trip[current] = (
                    trip.get(current, 0.0)
                    + delivery
                )

                remaining[current] -= delivery

                remaining_capacity -= delivery


            if remaining_capacity <= EPS:
                break


            candidates = [
                f
                for f, qty in remaining.items()
                if qty > EPS
                and f not in trip
            ]


            if not candidates:
                break


            # geographically closest next factory
            current = min(
                candidates,
                key=lambda f:
                    distance_matrix.loc[
                        current,
                        f
                    ]
            )


        groups.append(
            {
                factory: round(qty, 3)
                for factory, qty
                in trip.items()
            }
        )


    return groups


quantum_trip_groups = (
    create_multifactory_groups(
        residual,
        DIST,
        capacity=TRUCK_CAPACITY,
        max_factories=MAX_QUANTUM_FACTORIES
    )
)


print("=" * 70)
print("MULTI-FACTORY QUANTUM TRIP GROUPS")
print("=" * 70)

for i, trip in enumerate(
    quantum_trip_groups,
    start=1
):

    print(
        f"\nQuantum Trip {i}"
    )

    print(
        "Total load:",
        round(sum(trip.values()), 3),
        "tonnes"
    )

    for factory, qty in trip.items():

        print(
            f"   {factory:<12}: "
            f"{qty:.3f} t"
        )


print(
    "\nTotal trips =",
    len(single_trips)
    + len(quantum_trip_groups)
)

MULTI-FACTORY QUANTUM TRIP GROUPS

Quantum Trip 1
Total load: 10.0 tonnes
   Ragati      : 8.380 t
   Chema       : 0.848 t
   Nguguini    : 0.772 t

Quantum Trip 2
Total load: 10.0 tonnes
   Thunguri    : 7.170 t
   Mukangu     : 2.830 t

Quantum Trip 3
Total load: 10.0 tonnes
   Kiangai     : 6.489 t
   Kianjega    : 3.511 t

Quantum Trip 4
Total load: 3.7 tonnes
   Nguguini    : 2.320 t
   Kianjega    : 1.011 t
   Mukangu     : 0.369 t

Total trips = 24


In [7]:
# ============================================================
# ROUTE METRICS
# ============================================================

def route_metrics(
    order,
    deliveries
):
    """
    Calculate exact:
      - route distance
      - route time
      - load-dependent tonne-km
      - KSh transport cost

    Empty return has zero tonne-km cost.
    """

    load = sum(
        deliveries.values()
    )

    current = DEPOT

    total_distance = 0.0

    total_time = 0.0

    total_tonne_km = 0.0

    total_cost = 0.0

    legs = []


    for factory in order:

        d = float(
            DIST.loc[
                current,
                factory
            ]
        )

        t = float(
            TIME.loc[
                current,
                factory
            ]
        )


        leg_tonne_km = (
            load * d
        )

        leg_cost = (
            TRANSPORT_RATE
            * leg_tonne_km
        )


        legs.append({
            "From": current,
            "To": factory,
            "Load_tonnes": load,
            "Distance_km": d,
            "Time_hours": t,
            "Tonne_km": leg_tonne_km,
            "Cost_KSh": leg_cost,
            "Delivered_tonnes":
                deliveries[factory]
        })


        total_distance += d

        total_time += t

        total_tonne_km += (
            leg_tonne_km
        )

        total_cost += (
            leg_cost
        )


        # unload fertilizer
        load -= deliveries[
            factory
        ]


        current = factory


    # ========================================================
    # RETURN TO KARATINA
    # ========================================================

    if order:

        d = float(
            DIST.loc[
                current,
                DEPOT
            ]
        )

        t = float(
            TIME.loc[
                current,
                DEPOT
            ]
        )


        # Remaining load should normally be zero
        leg_tonne_km = (
            load * d
        )

        leg_cost = (
            TRANSPORT_RATE
            * leg_tonne_km
        )


        legs.append({
            "From": current,
            "To": DEPOT,
            "Load_tonnes": load,
            "Distance_km": d,
            "Time_hours": t,
            "Tonne_km": leg_tonne_km,
            "Cost_KSh": leg_cost,
            "Delivered_tonnes": 0.0
        })


        total_distance += d

        total_time += t

        total_tonne_km += leg_tonne_km

        total_cost += leg_cost


    return {
        "order": list(order),
        "distance_km":
            total_distance,
        "time_hours":
            total_time,
        "tonne_km":
            total_tonne_km,
        "cost_ksh":
            total_cost,
        "remaining_load":
            load,
        "legs":
            legs
    }

In [8]:
# ============================================================
# EXACT SMALL-SUBPROBLEM BENCHMARK
# ============================================================

def exact_route_benchmark(
    deliveries
):

    factories = list(
        deliveries.keys()
    )

    best = None


    for order in permutations(
        factories
    ):

        metrics = route_metrics(
            order,
            deliveries
        )


        # route-time constraint
        if (
            metrics["time_hours"]
            > MAX_ROUTE_TIME
            + EPS
        ):
            continue


        if (
            best is None
            or metrics["cost_ksh"]
            < best["cost_ksh"]
        ):

            best = metrics


    return best

In [9]:
# ============================================================
# EXACT SMALL-SUBPROBLEM BENCHMARK
# ============================================================

def exact_route_benchmark(
    deliveries
):

    factories = list(
        deliveries.keys()
    )

    best = None


    for order in permutations(
        factories
    ):

        metrics = route_metrics(
            order,
            deliveries
        )


        # route-time constraint
        if (
            metrics["time_hours"]
            > MAX_ROUTE_TIME
            + EPS
        ):
            continue


        if (
            best is None
            or metrics["cost_ksh"]
            < best["cost_ksh"]
        ):

            best = metrics


    return best

In [10]:
# ============================================================
# BINARY POLYNOMIAL UTILITIES
# ============================================================

def add_polynomial_term(
    polynomial,
    coefficient,
    variables
):
    """
    Binary variables satisfy x^2 = x,
    therefore duplicate variable indices collapse.
    """

    key = tuple(
        sorted(
            set(variables)
        )
    )

    polynomial[key] += float(
        coefficient
    )

In [11]:
# ============================================================
# CUBO -> ISING HAMILTONIAN
# ============================================================

def cubo_to_ising(
    polynomial,
    n_qubits
):
    """
    Convert arbitrary binary polynomial
    up to cubic order into a diagonal
    SparsePauliOp Hamiltonian.

    x = (1 - Z) / 2
    """

    ising_terms = defaultdict(
        float
    )


    for variables, coefficient in (
        polynomial.items()
    ):

        degree = len(
            variables
        )


        # Constant term
        if degree == 0:

            ising_terms[
                tuple()
            ] += coefficient

            continue


        # Expand:
        #
        # product x_i =
        # (1 / 2^degree)
        # product (1 - Z_i)

        for subset_size in range(
            degree + 1
        ):

            for subset in combinations(
                variables,
                subset_size
            ):

                sign = (
                    -1
                    if subset_size % 2
                    else 1
                )


                term_coefficient = (
                    coefficient
                    * sign
                    / (2 ** degree)
                )


                key = tuple(
                    sorted(subset)
                )

                ising_terms[key] += (
                    term_coefficient
                )


    # ========================================================
    # CONVERT TO PAULI LABELS
    # ========================================================

    labels = []

    coefficients = []


    for qubits, coefficient in (
        ising_terms.items()
    ):

        if abs(coefficient) < 1e-12:
            continue


        label = [
            "I"
        ] * n_qubits


        # Qiskit Pauli string order is reversed
        for q in qubits:

            label[
                n_qubits
                - 1
                - q
            ] = "Z"


        labels.append(
            "".join(label)
        )

        coefficients.append(
            coefficient
        )


    operator = SparsePauliOp(
        labels,
        coeffs=np.asarray(
            coefficients,
            dtype=complex
        )
    )


    return operator.simplify()

In [12]:
# ============================================================
# DECODING FUNCTIONS
# ============================================================

def bits_from_bitstring(
    bitstring,
    n_qubits
):

    bitstring = str(
        bitstring
    ).replace(
        " ",
        ""
    ).zfill(
        n_qubits
    )


    # reverse so bits[q] = qubit q
    return [
        int(bit)
        for bit
        in bitstring[::-1]
    ]


def assignment_is_feasible(
    bits,
    cluster,
    variable_index
):

    m = len(
        cluster
    )


    # Every factory once
    for factory in cluster:

        count = sum(
            bits[
                variable_index[
                    (factory, p)
                ]
            ]
            for p in range(m)
        )

        if count != 1:
            return False


    # Every position once
    for p in range(m):

        count = sum(
            bits[
                variable_index[
                    (factory, p)
                ]
            ]
            for factory in cluster
        )

        if count != 1:
            return False


    return True


def decode_order(
    bits,
    cluster,
    variable_index
):

    order = []

    m = len(
        cluster
    )


    for p in range(m):

        selected = [

            factory

            for factory in cluster

            if bits[
                variable_index[
                    (factory, p)
                ]
            ] == 1
        ]


        if len(selected) != 1:

            return None


        order.append(
            selected[0]
        )


    return order

In [14]:
# ============================================================
# EXTRACT FEASIBLE QUANTUM SAMPLES
# ============================================================

def extract_feasible_samples(
    qaoa_result,
    cluster,
    variable_index,
    n_qubits,
    deliveries
):

    distribution = (
        qaoa_result.eigenstate
    )


    if distribution is None:

        return []


    # Convert to binary-probability dictionary
    if hasattr(
        distribution,
        "binary_probabilities"
    ):

        probabilities = (
            distribution.binary_probabilities(
                num_bits=n_qubits
            )
        )

    else:

        probabilities = {}

        for state, probability in (
            distribution.items()
        ):

            if isinstance(
                state,
                str
            ):

                bitstring = (
                    state.zfill(
                        n_qubits
                    )
                )

            else:

                bitstring = format(
                    int(state),
                    f"0{n_qubits}b"
                )

            probabilities[
                bitstring
            ] = probability


    candidates = []


    for (
        bitstring,
        probability
    ) in probabilities.items():

        bits = bits_from_bitstring(
            bitstring,
            n_qubits
        )


        if not assignment_is_feasible(
            bits,
            cluster,
            variable_index
        ):

            continue


        order = decode_order(
            bits,
            cluster,
            variable_index
        )


        if order is None:
            continue


        metrics = route_metrics(
            order,
            deliveries
        )


        # Enforce 8-hour route constraint
        if (
            metrics["time_hours"]
            > MAX_ROUTE_TIME + EPS
        ):

            continue


        candidates.append({
            "bitstring":
                bitstring,
            "probability":
                float(probability),
            "order":
                order,
            "metrics":
                metrics
        })


    return candidates

In [17]:
# ============================================================
# QAOA SOLVER
# ============================================================

def solve_trip_with_qaoa(
    deliveries,
    reps=QAOA_REPS,
    shots=SHOTS,
    maxiter=MAXITER,
    seeds=RANDOM_SEEDS
):

    cluster = list(
        deliveries.keys()
    )


    # ========================================================
    # SINGLE FACTORY:
    # NO QUANTUM OPTIMIZATION NECESSARY
    # ========================================================

    if len(cluster) == 1:

        order = cluster

        metrics = route_metrics(
            order,
            deliveries
        )

        return {
            "method":
                "Direct single factory",
            "order":
                order,
            "metrics":
                metrics,
            "qubits":
                0,
            "quantum_energy":
                None,
            "probability":
                1.0,
            "fallback":
                False,
            "qaoa_result":
                None
        }


    # ========================================================
    # BUILD CUBO
    # ========================================================

    model = build_route_cubo(
        deliveries
    )


    cluster = model[
        "cluster"
    ]

    variable_index = model[
        "variable_index"
    ]

    n_qubits = model[
        "n_qubits"
    ]

    polynomial = model[
        "polynomial"
    ]


    if n_qubits > 20:

        raise ValueError(
            f"{n_qubits} qubits requested. "
            "Reduce quantum cluster size."
        )


    # ========================================================
    # CUBO -> ISING
    # ========================================================

    Hamiltonian = cubo_to_ising(
        polynomial,
        n_qubits
    )


    print("\n" + "=" * 70)

    print(
        "QAOA MULTI-FACTORY SUBPROBLEM"
    )

    print("=" * 70)

    print(
        "Factories:",
        cluster
    )

    print(
        "Deliveries:",
        deliveries
    )

    print(
        "Qubits:",
        n_qubits
    )

    print(
        "CUBO terms:",
        len(polynomial)
    )

    print(
        "Ising Pauli terms:",
        len(Hamiltonian)
    )


    best_quantum = None


    # ========================================================
    # MULTIPLE QAOA RESTARTS
    # ========================================================

    for restart, seed in enumerate(
        seeds,
        start=1
    ):

        print(
            f"\nQAOA restart {restart}"
            f" | seed = {seed}"
        )


        sampler = StatevectorSampler(
            default_shots=shots,
            seed=seed
        )


        classical_optimizer = (
            COBYLA(
                maxiter=maxiter
            )
        )


        rng = np.random.default_rng(
            seed
        )


        initial_point = (
            rng.uniform(
                low=0.0,
                high=np.pi,
                size=2 * reps
            )
        )


        qaoa = QAOA(
            sampler=sampler,
            optimizer=
                classical_optimizer,
            reps=reps,
            initial_point=
                initial_point,
            aggregation=
                CVAR_ALPHA
        )


        start_time = (
            time.perf_counter()
        )


        result = (
            qaoa.compute_minimum_eigenvalue(
                Hamiltonian
            )
        )


        elapsed = (
            time.perf_counter()
            - start_time
        )


        candidates = (
            extract_feasible_samples(
                result,
                cluster,
                variable_index,
                n_qubits,
                deliveries
            )
        )


        print(
            "Feasible sampled routes:",
            len(candidates)
        )


        if not candidates:

            continue


        # Find lowest true transport cost
        current_best = min(
            candidates,
            key=lambda x: (
                x["metrics"][
                    "cost_ksh"
                ],
                -x["probability"]
            )
        )


        current_best[
            "runtime_seconds"
        ] = elapsed

        current_best[
            "quantum_energy"
        ] = float(
            np.real(
                result.eigenvalue
            )
        )

        current_best[
            "qaoa_result"
        ] = result


        print(
            "Best sampled order:",
            current_best["order"]
        )

        print(
            "Cost:",
            f"KSh "
            f"{current_best['metrics']['cost_ksh']:,.2f}"
        )

        print(
            "Probability:",
            f"{current_best['probability']:.6f}"
        )


        if (
            best_quantum is None

            or current_best[
                "metrics"
            ][
                "cost_ksh"
            ]
            <
            best_quantum[
                "metrics"
            ][
                "cost_ksh"
            ]
        ):

            best_quantum = (
                current_best
            )


    # ========================================================
    # EXACT BENCHMARK
    # ========================================================

    exact = exact_route_benchmark(
        deliveries
    )


    # ========================================================
    # IF QAOA PRODUCED NO FEASIBLE SAMPLE
    # ========================================================

    fallback = False


    if best_quantum is None:

        print(
            "\nWARNING:"
            " QAOA produced no feasible"
            " sample."
        )

        print(
            "Using exact small-subproblem"
            " fallback."
        )

        fallback = True

        best_quantum = {
            "order":
                exact["order"],
            "metrics":
                exact,
            "probability":
                0.0,
            "runtime_seconds":
                np.nan,
            "quantum_energy":
                np.nan,
            "qaoa_result":
                None
        }


    quantum_cost = (
        best_quantum[
            "metrics"
        ][
            "cost_ksh"
        ]
    )


    approximation_ratio = (
        quantum_cost
        / exact["cost_ksh"]
        if exact["cost_ksh"] > 0
        else 1.0
    )


    print("\nExact route:", exact["order"])

    print(
        "Exact cost:",
        f"KSh {exact['cost_ksh']:,.2f}"
    )

    print(
        "QAOA approximation ratio:",
        f"{approximation_ratio:.6f}"
    )


    return {
        "method":
            "CUBO-QAOA",
        "order":
            best_quantum[
                "order"
            ],
        "metrics":
            best_quantum[
                "metrics"
            ],
        "qubits":
            n_qubits,
        "quantum_energy":
            best_quantum[
                "quantum_energy"
            ],
        "probability":
            best_quantum[
                "probability"
            ],
        "runtime_seconds":
            best_quantum[
                "runtime_seconds"
            ],
        "exact_order":
            exact["order"],
        "exact_cost":
            exact["cost_ksh"],
        "approximation_ratio":
            approximation_ratio,
        "fallback":
            fallback,
        "qaoa_result":
            best_quantum[
                "qaoa_result"
            ]
    }

In [18]:
test_trip = quantum_trip_groups[0]

test_result = solve_trip_with_qaoa(
    test_trip
)

print("\n")
print("=" * 70)
print("TEST QUANTUM RESULT")
print("=" * 70)

print(
    "Quantum route:",
    DEPOT,
    "->",
    " -> ".join(
        test_result["order"]
    ),
    "->",
    DEPOT
)

print(
    "Cost:",
    f"KSh "
    f"{test_result['metrics']['cost_ksh']:,.2f}"
)

print(
    "Distance:",
    f"{test_result['metrics']['distance_km']:.2f} km"
)

print(
    "Time:",
    f"{test_result['metrics']['time_hours']:.2f} hours"
)

print(
    "Qubits:",
    test_result["qubits"]
)

print(
    "Approximation ratio:",
    test_result[
        "approximation_ratio"
    ]
)

NameError: name 'build_route_cubo' is not defined